# Logit model

Suppose we are interested in predicting the probability of delisting of a stock due to either:
- Bankrupt
- Liquidation

as function of its size and other stock characteristics (leverage, ROA, cash).

We use two models:
1. OLS
2. Logit model

## 0. Required packages

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

## 1. Load and prepare data

We use data from Compustat. Delisting is stored as categorical variable `dlrsn`. Bankrupt and liquidation are coded as "02" and "03".

We create a balanced sample of delisted and active firms. In other words we randomly sample firms that delist due to bankrupt or liquidation and firms that do not in the next five years. 

Disclaimer: this may be seen as "cherry-picking". In fact, there is relative little variation in the sample for delisting firms. 

In [3]:
# ----------------------------
# Load Compustat quarterly extract
# Needs: gvkey, datadate, dldte, dlrsn, atq, ltq, niq, cheq
# ----------------------------
df = pd.read_csv("Delisting.csv", parse_dates=["datadate", "dldte"])

# Keep observations that are before deletion date (or never deleted)
df = df[df["dldte"].isna() | (df["datadate"] < df["dldte"])].copy()

# Clean deletion reason codes to 2-digit strings
df["dlrsn"] = df["dlrsn"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(2)

# ----------------------------
# Outcome: "failure-like deletion" within 1 year
# ----------------------------
FAIL_CODES = {"02", "03"}  # bankruptcy, liquidation (change if your coding differs)

days_to_del = (df["dldte"] - df["datadate"]).dt.days
df["FAIL_1Y"] = ((days_to_del >= 1) & (days_to_del <= 365) & (df["dlrsn"].isin(FAIL_CODES))).astype(int)

# Clean controls: survives at least 5 years
df["SURV_5Y"] = (df["dldte"].isna() | ((df["dldte"] - df["datadate"]).dt.days > 5 * 365)).astype(int)

# ----------------------------
# Multiple regressors (the four)
# ----------------------------
df = df[(df["atq"] > 0) & (df["ltq"] >= 0)].copy()

df["SIZE"] = np.log(df["atq"])
df["LEV"] = df["ltq"] / df["atq"]
df["ROA"] = df["niq"] / df["atq"]
df["CASH"] = df["cheq"] / df["atq"]

use_all = df[["FAIL_1Y", "SURV_5Y", "SIZE", "LEV", "ROA", "CASH"]].replace([np.inf, -np.inf], np.nan).dropna()

# Winsorize predictors (for stability/plot readability)
for c in ["SIZE", "LEV", "ROA", "CASH"]:
    lo, hi = use_all[c].quantile([0.001, 0.999])
    use_all[c] = use_all[c].clip(lo, hi)

# ----------------------------
# CHERRYPICK FOR TEACHING: balanced sample (events vs clean survivors)
# ----------------------------
events = use_all[use_all["FAIL_1Y"] == 1]
controls = use_all[(use_all["FAIL_1Y"] == 0) & (use_all["SURV_5Y"] == 1)]

n = len(events)
if n == 0:
    raise ValueError("No failure events found. Check FAIL_CODES, dldte coverage, and dlrsn coding.")

if len(controls) < n:
    controls_s = controls.sample(n=len(controls), random_state=0)
    events_s = events.sample(n=len(controls), random_state=0)
else:
    controls_s = controls.sample(n=n, random_state=0)
    events_s = events

use = pd.concat([events_s, controls_s], axis=0).sample(frac=1, random_state=1)  # shuffle

y = use["FAIL_1Y"].astype(float)
X = sm.add_constant(use[["SIZE", "LEV", "ROA", "CASH"]])

## 2. Estimate the model

The logit model reads as:

$$
p(X) = \frac{e^{\beta_0 + \beta_1 SIZE_i + \beta_2 LEV_i + \beta_3 CASH_i}}{1+ e^{\beta_0 + \beta_1 SIZE_i + \beta_2 LEV_i + \beta_3 CASH_i}}
$$

Additionally, we employ an OLS model for comparability.

The logit model can be estimate by Maximum Likelihood through the `statsmodels` package using `sm.Logit`, which includes the subfunction `fit()` (as we did for OLS).

Furthermore, recall that the logit fits a non-linear model: the interpretation of the coefficients must account for that. Thus we estimate:
- Average marginal effect (AME)
- Marginal effect at the mean (MEM)

Both can be estimated within the `Logit` function by calling `get_margeff` with argument `at` equal to `overall` and `mean` for AME and MEM, respectively.

In [4]:
# Logit (with robust SE in summary)
logit = sm.Logit(y, X).fit(disp=0)

print("\nBalanced sample event rate:", np.round(y.mean(), 3), "N:", len(use))

print("\n=== Logit ===")
print(logit.summary())


ame = logit.get_margeff(at='overall')  # AME
mem = logit.get_margeff(at='mean')  # MEM

print(ame.summary())
print(mem.summary())


Balanced sample event rate: 0.5 N: 2892

=== Logit ===
                           Logit Regression Results                           
Dep. Variable:                FAIL_1Y   No. Observations:                 2892
Model:                          Logit   Df Residuals:                     2887
Method:                           MLE   Df Model:                            4
Date:                Tue, 28 Apr 2026   Pseudo R-squ.:                 0.05108
Time:                        11:21:46   Log-Likelihood:                -1902.2
converged:                       True   LL-Null:                       -2004.6
Covariance Type:            nonrobust   LLR p-value:                 3.532e-43
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7496      0.093      8.060      0.000       0.567       0.932
SIZE          -0.1882      0.016    -11.761      0.000      -0.220      -0.